# Big Data – Graph Analysis with PySpark, GraphFrames & Neo4j

**Dataset:** Recommendation Videogames (Network Repository)  
**Tools:** PySpark · GraphFrames · Neo4j Connector

---

## 1. Dataset

Download the **Recommendation Videogames** dataset from the [Network Repository](https://networkrepository.com/rec-amazon-ratings-videogames.php).  
The dataset represents an Amazon product co-purchasing network for video games, where nodes are products/users and edges represent co-purchase or rating relationships.


In [ ]:
# Install dependencies
!pip install graphframes

import urllib.request, zipfile, os

# Download dataset from Network Repository
URL = "https://nrvis.com/download/data/rec/rec-amazon-ratings-Video_Games.zip"
ZIP_FILE = "rec-amazon-ratings-Video_Games.zip"
DATA_DIR = "data"

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(ZIP_FILE):
    print("Downloading dataset...")
    urllib.request.urlretrieve(URL, ZIP_FILE)
    print("Download complete.")

with zipfile.ZipFile(ZIP_FILE, 'r') as z:
    z.extractall(DATA_DIR)
    print("Files extracted:", z.namelist())

!ls -lh {DATA_DIR}/

---

## 2. Data Ingestion

Read the dataset using **PySpark** and build the vertices/edges DataFrames needed for GraphFrames.


In [ ]:
from pcamarillor.spark_utils import SparkUtils

neo4j_connector = (
    "org.neo4j:neo4j-connector-apache-spark_2.13:5.3.10_for_spark_3,"
    "io.graphframes:graphframes-spark3_2.13:0.9.0-spark3.5"
)

su = SparkUtils(
    "Recommendation Videogames Graph",
    "spark://spark-master:7077",
    spark_packages=neo4j_connector
)

spark = su.spark
spark

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

# ── Read raw edges file ──────────────────────────────────────────────────────
# The Network Repository edge-list format:
#   user_id  item_id  rating  timestamp
RAW_FILE = "data/rec-amazon-ratings-Video_Games.edges"

raw_schema = StructType([
    StructField("user_id",   StringType(), True),
    StructField("item_id",   StringType(), True),
    StructField("rating",    DoubleType(), True),
    StructField("timestamp", LongType(),   True),
])

raw_df = (
    spark.read
         .option("sep", " ")
         .option("comment", "%")
         .schema(raw_schema)
         .csv(RAW_FILE)
)

print(f"Total edges: {raw_df.count():,}")
raw_df.show(10, truncate=False)

In [ ]:
# ── Build Vertices DataFrame ──────────────────────────────────────────────────
# Collect all unique node IDs (users + items) and label their type
users = raw_df.select(F.col("user_id").alias("id")).withColumn("type", F.lit("User"))
items = raw_df.select(F.col("item_id").alias("id")).withColumn("type", F.lit("Product"))

vertices = users.union(items).distinct()

print(f"Total vertices: {vertices.count():,}")
vertices.show(10, truncate=False)

# ── Build Edges DataFrame ────────────────────────────────────────────────────
edges = (
    raw_df
    .withColumnRenamed("user_id", "src")
    .withColumnRenamed("item_id", "dst")
    .withColumn("relationship", F.lit("RATED"))
    .select("src", "dst", "relationship", "rating", "timestamp")
)

print(f"Total edges: {edges.count():,}")
edges.show(10, truncate=False)

---

## 3. Graph Analysis

Build a **GraphFrame** and run core graph algorithms.


In [ ]:
from graphframes import GraphFrame

g = GraphFrame(vertices, edges)

print("Vertices schema:")
g.vertices.printSchema()
print("Edges schema:")
g.edges.printSchema()

### 3.1 PageRank

PageRank measures the importance of each node based on the number and quality of incoming edges.

In [ ]:
pr_results = g.pageRank(resetProbability=0.15, maxIter=10)

pr_results.vertices.printSchema()

print("Top 20 nodes by PageRank (descending):")
(
    pr_results.vertices
    .select("id", "type", "pagerank")
    .orderBy("pagerank", ascending=False)
    .show(20, truncate=False)
)

### 3.2 Label Propagation

Label Propagation Algorithm (LPA) detects communities by propagating labels through the graph.

In [ ]:
lpa = g.labelPropagation(maxIter=5)

print("Label Propagation – community assignments:")
lpa.show(20, truncate=False)

print("Number of communities detected:", lpa.select("label").distinct().count())

### 3.3 Triangle Counting

Triangle counting finds the number of triangles each vertex belongs to, which is useful for measuring network clustering.

In [ ]:
triangle_count = g.triangleCount()

print("Triangle counts per vertex:")
(
    triangle_count
    .join(vertices, "id")
    .select("id", "type", "count")
    .orderBy("count", ascending=False)
    .show(20, truncate=False)
)

print("Total triangles in graph:", triangle_count.agg(F.sum("count")).collect()[0][0] // 3)

### 3.4 Degree Distribution

Degree distribution shows the connectivity structure of the graph — how many connections each node has.

In [ ]:
# ── In-Degree ────────────────────────────────────────────────────────────────
in_deg = g.inDegrees.join(vertices, "id")

print("In-Degree (top 20 – most rated products):")
in_deg.orderBy("inDegree", ascending=False).show(20, truncate=False)

print("In-Degree distribution summary:")
in_deg.select("inDegree").summary().show()

In [ ]:
# ── Out-Degree ───────────────────────────────────────────────────────────────
out_deg = g.outDegrees.join(vertices, "id")

print("Out-Degree (top 20 – most active reviewers):")
out_deg.orderBy("outDegree", ascending=False).show(20, truncate=False)

print("Out-Degree distribution summary:")
out_deg.select("outDegree").summary().show()

---

## 4. Writing Data in Neo4j

Persist the **vertices** (nodes) and **edges** (relationships) DataFrames into a Neo4j instance using the Neo4j Connector for Apache Spark.


In [ ]:
# ── Neo4j connection settings ─────────────────────────────────────────────────
NEO4J_URL    = "bolt://neo4j-iteso:7687"
NEO4J_USER   = "neo4j"
NEO4J_PASSWD = "neo4j@1234"

# ── Write Vertices (nodes) ────────────────────────────────────────────────────
(
    g.vertices.write
    .format("org.neo4j.spark.DataSource")
    .mode("Overwrite")
    .option("url",  NEO4J_URL)
    .option("authentication.basic.username", NEO4J_USER)
    .option("authentication.basic.password", NEO4J_PASSWD)
    .option("labels", ":Node")
    .option("node.keys", "id")
    .save()
)

print(f"{g.vertices.count():,} nodes written to Neo4j")

In [ ]:
# ── Write Edges (relationships) ───────────────────────────────────────────────
(
    g.edges.write
    .format("org.neo4j.spark.DataSource")
    .mode("Overwrite")
    .option("url",  NEO4J_URL)
    .option("authentication.basic.username", NEO4J_USER)
    .option("authentication.basic.password", NEO4J_PASSWD)
    .option("relationship", "RATED")
    .option("relationship.save.strategy", "keys")
    .option("relationship.source.labels", ":Node")
    .option("relationship.source.save.mode", "match")
    .option("relationship.source.node.keys", "src:id")
    .option("relationship.target.labels", ":Node")
    .option("relationship.target.save.mode", "match")
    .option("relationship.target.node.keys", "dst:id")
    .save()
)

print(f"{g.edges.count():,} relationships written to Neo4j")

---

## 5. Querying the Graph

After writing the graph to Neo4j, open the **Neo4j Browser** at `http://neo4j-iteso:7474` and run the following Cypher query to visualize the graph:

```cypher
MATCH (n:Node)-[r:RATED]->(m:Node)
RETURN n, r, m
LIMIT 100
```

The screenshot below shows the resulting graph visualization in Neo4j Browser:

> **📸 Screenshot placeholder** – Replace this cell with an image of your Neo4j Browser graph visualization.
> 
> To add your screenshot:
> 1. Run the Cypher query above in Neo4j Browser
> 2. Take a screenshot of the graph view
> 3. Save it as `neo4j_graph.png` in the same folder as this notebook
> 4. Uncomment and run the cell below


In [ ]:
# Display the Neo4j graph screenshot
from IPython.display import Image, display

display(Image(filename="neo4j_graph.png"))

In [ ]:
# Stop the Spark session
su.spark.stop()
print("Spark session stopped.")